## 1. Extração do Texto das Provas do ENEM

Foram desenvolvidos métodos padronizados para a extração de texto das provas do ENEM, utilizando ferramentas e bibliotecas Python, além de técnicas de OCR. 
No entanto, cada edição do exame apresentou particularidades que exigiram abordagens específicas. 

Assim, enquanto em alguns anos foi possível realizar a extração automática dos enunciados e alternativas — com posterior refinamento e revisão manual —, em outros anos foi necessário recorrer à extração manual.

In [2]:
# Dependências
import pymupdf
import re
import csv
import pandas as pd

### Extração 2009 - 2011

### Extração 2012 - 2014

Extração concluída **manualmente**.

### Extração 2015-2017

### Extração 2018-2020

### Extração 2021 (Enem Digital)

Em 2021, foi aplicada a edição digital do ENEM, que apresentou uma estrutura significativamente diferente da versão regular. Como, por alguma razão, a prova da aplicação regular foi disponibilizada em um formato PDF não extraível (com conteúdo criptografado), optamos por realizar a extração automática utilizando outro método no Enem Digital.

In [ ]:
input_path = f"../data/raw/provas/ENEM_2021.pdf"
output_path = f"../data/questions_extracted/enem_2021.pdf"

# Extraindo texto do ENEM Digital 2021
document = pymupdf.open(input_path)
total_text = ""

for page in document.pages(5, 53):
    text = page.get_text()
    total_text += text + "\n"

In [ ]:
# Padrão do ENEM Digital
question_pattern = r"Questão\s+(\d+)\s+-\s+Ciências da Natureza e suas Tecnologias(.*?)(?=Questão\s+\d+\s+-\s+Ciências da Natureza e suas Tecnologias|$)"
question_matches = re.findall(question_pattern, total_text, flags=re.DOTALL)

results = []

# Padrão de formatação das alternativas do ENEM Digital
alt_regex = r"\n(?:\d+\n)*([A-E])\n([^\n]+)"

for num_quest, block in question_matches:
    content = block.strip()

    alt_raw = re.findall(alt_regex, content)

    enunciado = content
    alt_match = re.search(alt_regex, content)
    if alt_match:
        enunciado = content[: alt_match.start()].strip()

    # Formatando alternativas como "A: texto"
    alternativas = "; ".join([f"{letter}: {text.strip()}" for letter, text in alt_raw])

    results.append(
        {
            "numero_questao": num_quest,
            "enunciado": enunciado,
            "alternativas": alternativas,
        }
    )

In [ ]:
# Salvando em CSV
with open(output_path, "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["numero_questao", "enunciado", "alternativas"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in results[1:]:
        writer.writerow(row)

print("Extração Finalizada")

In [ ]:
df = pd.read_csv(output_path)
df.head()

### Extração 2015-2018, 2022-2023 e 2019, 2020, 2022 e 2023 ledor

In [4]:
# Definição de Variáveis de Input e Output
start_year = 2022
end_year = 2023

In [5]:
# Iterando pelos anos 2022-2023
for year in range(start_year, end_year + 1):

    # Definindo input e output paths
    input_path = f"../data/raw/provas/ENEM_{year}.pdf"
    output_path = f"../data/questions_extracted/enem_{year}.csv"

    # Extraindo texto do ENEM
    document = pymupdf.open(input_path)
    total_text = ""

    for page in document.pages(1, 15):
        text = page.get_text()
        total_text += text + "\n"
    
    # Remover o rodapé
    rodape_pattern = r"CN\s+•\s+2º\s+DIA\s+•\s+CADERNO\s+7\s+•\s+AZUL"
    total_text = re.sub(rodape_pattern, "", total_text)

    # Remover códigos do rodapé, por exemplo: *020325AZ5*
    total_text = re.sub(r"\*\w+\*", "", total_text)

    # Remover linhas que contenham apenas dígitos (possíveis números de página)
    total_text = re.sub(r"\n\s*\d+\s*\n", "\n", total_text)
    total_text = re.sub(r"\n\s*\d+\s*$", "", total_text)

    # Dividindo o texto em blocos de questões
    question_blocks = re.split(r"QUESTÃO\s+", total_text)
    question_blocks = [block.strip() for block in question_blocks if block.strip()]

    results = []

    for block in question_blocks:
        # Extraindo número da questão
        m = re.match(r"(\d+)", block)
        if not m:
            continue 
        num_quest = m.group(1)
        
        content = block[m.end():].strip()
        
        # Separando enunciado e alternativas.
        parts = re.split(r"\nA\s*\n", content, maxsplit=1)
        if len(parts) == 2:
            enunciado = parts[0].strip()
            alt_text = "A\n" + parts[1].strip()
        else:
            enunciado = content
            alt_text = ""
        
        # Extraindo alternativas
        alt_regex = r"([A-E])\s+[A-E]\s+(.*?)(?=(?:\n[A-E]\s+[A-E]\s+)|\Z)"
        alt_raw = re.findall(alt_regex, alt_text, flags=re.DOTALL)
        
        # Formatando alternativas como "A: texto"
        alternativas = "; ".join([f"{letter}: {text.strip()}" for letter, text in alt_raw])
        
        results.append({
            "numero_questao": num_quest,
            "enunciado": enunciado,
            "alternativas": alternativas
        })

    # Salvando em CSV
    with open(output_path, "w", newline="", encoding="utf-8") as csvfile:
        fieldnames = ["numero_questao", "enunciado", "alternativas"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for row in results[1:]:
            writer.writerow(row)

    print(f"Extração de {year} finalizada")

Extração de 2022 finalizada
Extração de 2023 finalizada


# Extração 2018 e 2021 Ledor

In [ ]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io

years = (2017, 2021)

for year in years:
    input_path = f"../data/provas/ENEM_{year}_LEDOR.pdf"
    output_path = f"../data/extracted_questions/enem_{year}_LEDOR.csv"

    caminho_pdf = input_path
    doc = fitz.open(caminho_pdf)

    texto_total = ""

    for pagina in doc[1:16]:
        # Renderiza a página como imagem
        pix = pagina.get_pixmap(dpi=300)
        img = Image.open(io.BytesIO(pix.tobytes()))

        # OCR para extrair texto da imagem
        texto = pytesseract.image_to_string(img, lang='por')
        texto_total += texto + "\n"

    doc.close()

    # Remover o rodapé
    rodape_pattern = r"CN\s+•\s+2º\s+DIA\s+•\s+Caderno\s+11\s+•\s+LARANJA"
    full_text = re.sub(rodape_pattern, "", texto_total)

    # Remover códigos do rodapé, por exemplo: *020325AZ5*
    full_text = re.sub(r"\*\w+\*", "", full_text)

    # Remover linhas que contenham apenas dígitos (possíveis números de página)
    full_text = re.sub(r"\n\s*\d+\s*$", "", full_text)

    # Dividindo o texto em blocos de questões
    question_blocks = re.split(r"QUESTÃO\s+", full_text)
    question_blocks = [block.strip() for block in question_blocks if block.strip()]

    results = []

    for block in question_blocks:
        # Extraindo número da questão
        m = re.match(r"(\d+)", block)
        if not m:
            continue 
        num_quest = m.group(1)
        
        content = block[m.end():].strip()

        content = content.replace('\x03', '').replace('\x0f', '').replace('\x11', '')
        
        # Separando enunciado e alternativas.
        padrao = r"(\nA\t.*\nB\t.*\nC\t.*\nD\t.*\nE\t.*)"
        match = re.search(padrao, content, flags=re.DOTALL)
        if match:
            enunciado = content[:match.start()].strip()
            alt_text = match.group(0).strip()
        else:
            enunciado = content
            alt_text = ""
        
        # Extraindo alternativas
        alt_regex = r"([A-E])\s+(.*?)(?=\n[A-E]\s+|$)"
        alt_raw = re.findall(alt_regex, alt_text, flags=re.DOTALL)
        
        # Formatando alternativas como "A: texto"
        alternativas = "; ".join([f"{letter}: {text.strip()}" for letter, text in alt_raw])
        results.append({
            "numero_questao": num_quest,
            "enunciado": enunciado,
            "alternativas": alternativas
        })

    with open(output_path, "w", newline="", encoding="utf-8") as csvfile:
        fieldnames = ["numero_questao", "enunciado", "alternativas"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for row in results:
            writer.writerow(row)